In [1]:
import pandas as pd

In [ ]:
df=pd.read_csv("../data/processed/train.csv",low_memory=False)

In [ ]:
df=df.drop(columns=["Label"])

In [ ]:
import numpy as np

In [ ]:

corr_mat=df.corr().abs()
upper=corr_mat.where(np.triu(np.ones(corr_mat.shape),k=1).astype(bool))
high_corr=upper.stack()
high_corr=high_corr[(high_corr>0.85)]
print(high_corr.sort_values(ascending=False))

In [ ]:
for col in ['ECE Flag Cnt','CWE Flag Count','Active Max','Idle Max','Active Mean']:
    print(col,df[col].nunique(),df[col].std())

In [ ]:
print((df['Active Max']==df['Idle Max']).mean())

In [ ]:
df=pd.read_csv("../data/raw/API/API.csv")
print((df['Active Max']==df['Idle Max']).mean())

In [ ]:
def find_duplicate_column(df):
    duplicate=[]
    checked=set()
    cols=df.columns.tolist()
    for i,col1 in enumerate(cols):
        if col1 in checked :
            continue
        group=[col1]
        for col2 in cols[i+1:]:
            if col2 in checked :
                continue
            if df[col1].equals(df[col2]):
                group.append(col2)
                checked.add(col2)
            if len(group) >1:
                duplicate.append(group)
                checked.add(col1)
    return duplicate

train=pd.read_csv("../data/processed/train.csv")
val=pd.read_csv("../data/processed/validation.csv")
test=pd.read_csv("../data/processed/test.csv")

cols_to_drop={}
i=0
for d in [train,test,val]:
    d['Label']=d['Label'].str.strip().str.lower()
    """
    unnamed_cols=[c for c in d.columns if c.startswith('Unnamed:')]
    d.drop(columns=unnamed_cols,inplace=True,errors='ignore')
"""
    dup_groups=find_duplicate_column(d)
    cols_to_drop[i]={col for group in dup_groups for col in group[1:]}
    i+=1
all_col=list(cols_to_drop.values())
union=set.union(*all_col)
intersection=set.intersection(*all_col)
dis=union-intersection
print(dis)

"""
train=train.drop(columns=cols_to_drop)
test=test.drop(columns=cols_to_drop)
val=val.drop(columns=cols_to_drop)
"""

In [ ]:
print(sorted(train['Label'].unique()))

In [2]:
label_col="Label"

train=pd.read_csv("../data/processed/train.csv")
test=pd.read_csv("../data/processed/test.csv")

X_train=train.drop(columns=[label_col])
y_train =train[label_col]

X_test=test.drop(columns=[label_col])
y_test=test[label_col]




In [3]:
from sklearn.preprocessing import LabelEncoder


In [4]:

le=LabelEncoder()
y_train=le.fit_transform(y_train)
y_test=le.transform(y_test)

In [6]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

In [ ]:
clf=RandomForestClassifier(n_estimators=200,max_depth=15,random_state=42,n_jobs=1,class_weight='balanced')
clf.fit(X_train,y_train)

In [ ]:
y_pred=clf.predict(X_test)
print(classification_report(y_test,y_pred,target_names=le.classes_))

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay,confusion_matrix
import matplotlib.pyplot as plt

In [ ]:
fig,ax=plt.subplots(figsize=(14,12))
ConfusionMatrixDisplay.from_predictions(y_val,y_pred,xticks_rotation=90,ax=ax)
plt.show()

In [ ]:
y_test_pred=clf.predict(X_test)
print(classification_report(y_test,y_test_pred,target_names=le.classes_))

In [ ]:
print(train['Label'].value_counts()[['dos','slowloris']])
print(test['Label'].value_counts()[['dos','slowloris']])

In [ ]:
cm_test=confusion_matrix(y_test,y_test_pred)
cm_df=pd.DataFrame(cm_test,index=le.classes_,columns=le.classes_)
print(cm_df.loc[['dos','slowloris']])

In [5]:
import xgboost as xgb

In [7]:
from sklearn.metrics import classification_report

In [ ]:
clf=xgb.XGBClassifier(
    objective="multi:softprob",
    num_class=len(le.classes_),
    max_depth=6,
    learning_rate=0.1,
    n_estimators=300,
    subsample=0.8,
    colsample_bytree=0.8,
    tree_method="hist",
    eval_metric="mlogloss",
    n_jobs=-1,
    random_state=42
)
clf.fit(X_train,y_train)
y_pred=clf.predict(X_test)
print(classification_report(y_test,y_pred,target_names=le.classes_.astype("str")))

In [6]:
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report


In [7]:
pca=PCA(n_components=15,random_state=72)
X_train_pca=pca.fit_transform(X_train)
X_test_pca=pca.transform(X_test)

In [ ]:
clf=RandomForestClassifier(random_state=72)
clf.fit(X_train_pca,y_train)
y_pred_pca=clf.predict(X_test_pca)

print(classification_report(y_test,y_pred_pca,target_names=le.classes_))

In [9]:

clf=xgb.XGBClassifier(
    objective="multi:softprob",
    num_class=len(le.classes_),
    max_depth=6,
    learning_rate=0.1,
    n_estimators=300,
    subsample=0.8,
    colsample_bytree=0.8,
    tree_method="hist",
    eval_metric="mlogloss",
    n_jobs=-1,
    random_state=42
)
clf.fit(X_train_pca,y_train)
y_pred=clf.predict(X_test_pca)
print(classification_report(y_test,y_pred,target_names=le.classes_.astype("str")))

                precision    recall  f1-score   support

           api       0.82      0.66      0.73      4020
        benign       0.89      0.97      0.93    128620
    bruteforce       0.86      0.65      0.74      5286
bufferoverflow       0.83      0.55      0.67      4616
        c2beac       0.94      0.82      0.88      5933
          ddos       1.00      1.00      1.00      9441
           dns       0.89      0.80      0.85      8443
           dos       0.77      0.67      0.72      7500
       evasion       0.87      0.29      0.43      4271
  exfiltracion       0.93      0.94      0.93      5588
   explotacion       0.63      0.40      0.49      7529
          mitm       0.99      0.96      0.97      5307
     slowloris       0.62      0.73      0.67      5439
        tlsssl       0.79      0.78      0.78      6107
      webbased       0.84      0.75      0.79      7698

      accuracy                           0.87    215798
     macro avg       0.84      0.73      0.77 